# search

In [14]:
import asyncio
from src.search import match_product

In [22]:

result = await match_product(
    product_name="Volvic Mineral Water 12x1L",
    website="amazon.co.uk",
    country="uk",
)
print(result.verdict)                          # FinalVerdict.MATCH / NO_MATCH
print(result.matched_candidate.url if result.matched_candidate else "not found")
print(result.layer_trace.to_dict())            # per-layer pass/fail/unknown
print(result.reason)                           # LLM rationale or pipeline statusdemo())

FinalVerdict.MATCH
https://www.amazon.co.uk/Volvic-Natural-Mineral-Water-12/dp/B07G9GVMD3
{'domain': 'pass', 'brand': 'pass', 'numeric': 'pass', 'distinguishing': 'pass'}
Candidate 1 is the same 12x1L Volvic Natural Mineral Water product. (via duckduckgo)


# scraping

In [1]:
import asyncio
from src.scraping import scrape
from pprint import pprint

In [2]:
url_1 = "https://www.argos.co.uk/product/3284476"
result = await scrape(url_1)

pprint(vars(result))

{'availability_raw': 'Available credit options',
 'brand': 'Forest Garden',
 'currency': 'GBP',
 'gtin': '5013050000000',
 'image_urls': ['https://media.4rgos.it/i/Argos/3284476_R_Z001A',
                'https://media.4rgos.it/i/Argos/3284476_R_Z002A',
                'https://media.4rgos.it/i/Argos/3284476_R_Z003A',
                'https://media.4rgos.it/i/Argos/3284476_R_Z004A'],
 'in_stock': True,
 'list_price': None,
 'membership_price': None,
 'parser_version': 'cs_20260818_201425',
 'price': Decimal('58.00'),
 'raw': None,
 'scraped_at': datetime.datetime(2026, 8, 21, 20, 58, 33, 616167, tzinfo=datetime.timezone.utc),
 'source_type': 'html',
 'title': 'Forest Pressure Treated Wooden Shed Base - 6 x 3ft',
 'url': 'https://www.argos.co.uk/product/3284476',
 'variant': None,
 'website': 'argos'}


# match

##  load the image


In [5]:
image_urls = ['https://images3.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-com/t1/4294967296/5636096/61097966189988/1139760/68e54592E52b01a56/4c765c6ab87a3a67.png.webp']

In [6]:
from image_load_compression import normalize_batch, load_config

results = await normalize_batch(image_urls, load_config(), run_id="nb")


In [8]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

def save_results(results, output_dir="output", run_id=None):
    """按 CLI 的约定落盘：图片 + results.jsonl"""
    run_id = run_id or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    target = Path(output_dir) / run_id
    target.mkdir(parents=True, exist_ok=True)

    for r in results:
        if r.image_bytes:
            digest = hashlib.sha256(r.url.encode()).hexdigest()[:16]
            ext = {"JPEG": "jpg"}.get(r.output_format or "", (r.output_format or "bin").lower())
            (target / f"{digest}.{ext}").write_bytes(r.image_bytes)

    with (target / "results.jsonl").open("w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")
    return target

save_results(results, output_dir="output", run_id="nb")


PosixPath('output/nb')